In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from google.colab import files


df = pd.read_csv("Myntra_Fashion.csv")

<ipython-input-26-ee42689b6794>:8: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Myntra_Fashion.csv")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

# Define the neural network architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(784, 1024)
        self.fc5 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc5(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc4(x))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)

augmentation_transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=10,  # Random rotation between -10 and 10 degrees
        translate=(0.1, 0.1),  # Random translation up to 10% of image size
        scale=(0.9, 1.1),  # Random scaling between 90% and 110%
        shear=10  # Random shear up to 10 degrees
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # Keep MNIST normalization
])

# Download and load the MNIST dataset
transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
        ])

train_dataset = datasets.MNIST('../data', train=True, download=True, transform=augmentation_transform)
# train_dataset = datasets.MNIST('../data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('../data', train=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=True)

# Initialize the model, optimizer, and loss function
model = Net()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_function = nn.NLLLoss()

# Training loop
epochs = 7
for epoch in range(epochs):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = loss_function(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))


# Testing the model
correct = 0
with torch.no_grad():
  for data, target in test_loader:
    output = model(data)
    pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
    correct += pred.eq(target.view_as(pred)).sum().item()

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
    correct, len(test_loader.dataset),
    100. * correct / len(test_loader.dataset)))

100%|██████████| 9.91M/9.91M [00:00<00:00, 56.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.65MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.82MB/s]


Train Epoch: 0 [0/60000 (0%)]	Loss: 2.294146
Train Epoch: 0 [6400/60000 (11%)]	Loss: 0.847835
Train Epoch: 0 [12800/60000 (21%)]	Loss: 0.521610
Train Epoch: 0 [19200/60000 (32%)]	Loss: 0.531051
Train Epoch: 0 [25600/60000 (43%)]	Loss: 0.407804
Train Epoch: 0 [32000/60000 (53%)]	Loss: 0.218942
Train Epoch: 0 [38400/60000 (64%)]	Loss: 0.381005
Train Epoch: 0 [44800/60000 (75%)]	Loss: 0.449100
Train Epoch: 0 [51200/60000 (85%)]	Loss: 0.227822
Train Epoch: 0 [57600/60000 (96%)]	Loss: 0.113225
Train Epoch: 1 [0/60000 (0%)]	Loss: 0.102317
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.307548
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.278070
Train Epoch: 1 [19200/60000 (32%)]	Loss: 0.145481
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.056201
Train Epoch: 1 [32000/60000 (53%)]	Loss: 0.186198
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.327481
Train Epoch: 1 [44800/60000 (75%)]	Loss: 0.081613
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.454191
Train Epoch: 1 [57600/60000 (96%)]	Loss: 0.057799
Train Epoch:

In [ ]:
# prompt: use the above network to identify a number uploaded by the user

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np


# Function to preprocess the uploaded image
def preprocess_image(image_path):
    # Open the image
    image = Image.open(image_path).convert("L")  # Convert to grayscale
    # Resize the image to 28x28
    image = image.resize((28, 28))
    # Convert image to numpy array and normalize
    image = np.array(image).astype(np.float32) / 255.0
    # Invert the colors (if needed, assuming the MNIST model expects white digits on black background)
    image = 1.0 - image
    # Convert to tensor and add batch and channel dimensions
    tensor = torch.Tensor(image).unsqueeze(0).unsqueeze(0)
    # Normalize the tensor (mean and std from MNIST dataset)
    tensor = transforms.Normalize((0.1307,), (0.3081,))(tensor)
    return tensor

# Upload the image file
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# Preprocess the uploaded image
image_tensor = preprocess_image(image_path)


# Make a prediction
with torch.no_grad():
    output = model(image_tensor)
    predicted_digit = output.argmax(dim=1, keepdim=True).item()

print(f"Predicted digit: {predicted_digit}, from: {output}")

In [ ]:
def combine_features(row):
    return f"{row['BrandName']} {row['Category']} {row['category_by_Gender']} {row['Description']}"

df["combined_text"] = df.apply(combine_features, axis=1)
model= SentenceTransformer("all-MiniLm-L6-v2")
#embeddings = model.encode(df["combined_text"].tolist(), show_progress_bar=True)
embeddings=np.load("embeddings.npy")
def get_top_k_similar(query_text, k=5):
    query_embedding = model.encode([query_text])
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:k]
    return df.iloc[top_indices][["URL"]]
#np.save("embeddings.npy", embeddings)
query = "sangria handbag"

results = get_top_k_similar(query)
pd.set_option('display.max_colwidth', None)
print(results)

                                                                                                                           URL
435770  https://www.myntra.com/handbags/sangria/sangria-red--pink-geometric-self-design-shoulder-bag-with-tassell/16070452/buy
436044               https://www.myntra.com/handbags/sangria/sangria-blue--red-ethnic-motifs-printed-shoulder-bag/16070474/buy
435883      https://www.myntra.com/handbags/sangria/sangria-blue--grey-floral-printed-shoulder-bag-with-tasselled/16070438/buy
435956               https://www.myntra.com/handbags/sangria/sangria-navy-blue--white-floral-printed-shoulder-bag/16070454/buy
436045       https://www.myntra.com/handbags/sangria/sangria-coral-pink--white-ethnic-motifs-printed-shoulder-bag/16070462/buy


In [ ]:
import pandas as pd
import os
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings

df = pd.read_csv("test_csv2.csv")

def combine_row_to_document(row):
    combined_text = f"{row['brand_name']} {row['category']} {row['individual_category']} {row['description']} {row['size_option']}"
    metadata = {
        "brand_name": row["brand_name"],
        "category": row["category"],
        "individual_category": row["individual_category"],
        "description": row["description"],
        "size_option": row["size_option"]
    }
    return Document(page_content=combined_text, metadata=metadata)

documents = [combine_row_to_document(row) for _, row in df.iterrows()]

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(documents, embedding)

save_path = "/content/test_faiss_index"
os.makedirs(save_path)
vectorstore.save_local(save_path)
print("ok")

ok


In [ ]:
import pandas as pd
df = pd.read_csv("myntra_dataset.csv")
unique_categories = df['individual_category'].unique()
print(f"\nNumber of unique individual categories: {len(unique_categories)}")
print("\nUnique individual categories:")
for category in sorted(unique_categories):
            print(f"- {category}")


Number of unique individual categories: 84

Unique individual categories:
- baby-dolls
- bath-robe
- blazers
- boots
- bra
- bracelet
- briefs
- burqas
- camisoles
- capris
- casual-shoes
- churidar
- clothing-set
- co-ords
- coats
- dhotis
- dress-material
- dresses
- dungarees
- dupatta
- earrings
- flats
- flip-flops
- hair-accessory
- handbags
- harem-pants
- heels
- jackets
- jeans
- jeggings
- jewellery-set
- jumpsuit
- kurta-sets
- kurtas
- kurtis
- leggings
- lehenga-choli
- lingerie-accessories
- lingerie-set
- lounge-pants
- lounge-shorts
- lounge-tshirts
- necklace-and-chains
- nehru-jackets
- night-suits
- nightdress
- outdoor-masks
- palazzos
- patiala
- patiala-and-dupatta
- pyjamas
- rain-jacket
- robe
- salwar
- salwar-and-dupatta
- saree-accessories
- saree-blouse
- sarees
- scarves
- shapewear
- shawl
- shirts
- shorts
- shrug
- skirts
- sleepsuit
- slips
- socks
- sports-sandals
- stockings
- stoles
- sweaters
- sweatshirts
- swimwear
- thermal-bottoms
- thermal-top

In [ ]:
import pandas as pd
category_seasons = {
    "baby-dolls": ["summer"],
    "camisoles": ["summer"],
    "capris": ["summer"],
    "casual-shoes": ["summer", "spring", "fall", "winter", "all-season"],
    "dresses": ["summer", "spring"],
    "flip-flops": ["summer"],
    "handbags": ["summer", "spring", "fall", "winter", "all-season"],
    "harem-pants": ["summer", "spring"],
    "shorts": ["summer"],
    "skirts": ["summer", "spring"],
    "sleepsuit": ["summer"],
    "swimwear": ["summer"],
    "tshirts": ["summer", "spring"],
    "tops": ["summer", "spring"],
    "blazers": ["winter", "fall"],
    "boots": ["winter", "fall"],
    "coats": ["winter"],
    "jackets": ["winter", "fall"],
    "nehru-jackets": ["winter", "fall"],
    "shawl": ["winter", "fall"],
    "sweaters": ["winter", "fall"],
    "sweatshirts": ["winter", "fall"],
    "thermal-bottoms": ["winter"],
    "thermal-tops": ["winter"],
    "waistcoat": ["winter", "fall"],
    "rain-jacket": ["monsoon"],
    "bra": ["summer", "spring", "fall", "winter", "all-season"],
    "bracelet": ["summer", "spring", "fall", "winter", "all-season"],
    "briefs": ["summer", "spring", "fall", "winter", "all-season"],
    "burqas": ["summer", "spring", "fall", "winter", "all-season"],
    "churidar": ["summer", "spring", "fall", "winter", "all-season"],
    "clothing-set": ["summer", "spring", "fall", "winter", "all-season"],
    "co-ords": ["summer", "spring", "fall", "winter", "all-season"],
    "dhotis": ["summer", "spring", "fall", "winter", "all-season"],
    "dress-material": ["summer", "spring", "fall", "winter", "all-season"],
    "dungarees": ["spring", "fall"],
    "dupatta": ["summer", "spring", "fall", "winter", "all-season"],
    "earrings": ["summer", "spring", "fall", "winter", "all-season"],
    "flats": ["summer", "spring", "fall", "winter", "all-season"],
    "hair-accessory": ["summer", "spring", "fall", "winter", "all-season"],
    "heels": ["summer", "spring", "fall", "winter", "all-season"],
    "jeans": ["summer", "spring", "fall", "winter", "all-season"],
    "jeggings": ["summer", "spring", "fall", "winter", "all-season"],
    "jewellery-set": ["summer", "spring", "fall", "winter", "all-season"],
    "jumpsuit": ["summer", "spring", "fall"],
    "kurta-sets": ["summer", "spring", "fall", "winter", "all-season"],
    "kurtas": ["summer", "spring", "fall", "winter", "all-season"],
    "kurtis": ["summer", "spring", "fall", "winter", "all-season"],
    "leggings": ["summer", "spring", "fall", "winter", "all-season"],
    "lehenga-choli": ["summer", "spring", "fall", "winter", "all-season"],
    "lingerie-accessories": ["summer", "spring", "fall", "winter", "all-season"],
    "lingerie-set": ["summer", "spring", "fall", "winter", "all-season"],
    "lounge-pants": ["summer", "spring", "fall", "winter", "all-season"],
    "lounge-shorts": ["summer", "spring"],
    "lounge-tshirts": ["summer", "spring", "fall", "winter", "all-season"],
    "necklace-and-chains": ["summer", "spring", "fall", "winter", "all-season"],
    "night-suits": ["summer", "spring", "fall", "winter", "all-season"],
    "nightdress": ["summer", "spring", "fall", "winter", "all-season"],
    "outdoor-masks": ["summer", "spring", "fall", "winter", "all-season"],
    "palazzos": ["summer", "spring", "fall"],
    "patiala": ["summer", "spring", "fall", "winter", "all-season"],
    "patiala-and-dupatta": ["summer", "spring", "fall", "winter", "all-season"],
    "pyjamas": ["summer", "spring", "fall", "winter", "all-season"],
    "robe": ["winter", "fall"],
    "bath-robe": ["winter", "fall"],
    "salwar": ["summer", "spring", "fall", "winter", "all-season"],
    "salwar-and-dupatta": ["summer", "spring", "fall", "winter", "all-season"],
    "saree-accessories": ["summer", "spring", "fall", "winter", "all-season"],
    "saree-blouse": ["summer", "spring", "fall", "winter", "all-season"],
    "sarees": ["summer", "spring", "fall", "winter", "all-season"],
    "scarves": ["winter", "fall"],
    "shapewear": ["summer", "spring", "fall", "winter", "all-season"],
    "shirts": ["summer", "spring", "fall", "winter", "all-season"],
    "shrug": ["spring", "fall"],
    "slips": ["summer", "spring", "fall", "winter", "all-season"],
    "socks": ["summer", "spring", "fall", "winter", "all-season"],
    "sports-sandals": ["summer", "spring"],
    "stockings": ["winter", "fall"],
    "stoles": ["winter", "fall"],
    "tights": ["winter", "fall"],
    "track-pants": ["winter", "fall"],
    "tracksuits": ["winter", "fall"],
    "trousers": ["summer", "spring", "fall", "winter", "all-season"],
    "tunics": ["spring", "fall"],
}
category_weather = {
    "baby-dolls": ["hot"],
    "camisoles": ["hot"],
    "capris": ["hot", "warm"],
    "casual-shoes": ["sunny", "rainy", "cold", "cool", "warm", "hot", "all-weather"],
    "dresses": ["hot", "warm"],
    "flip-flops": ["hot", "sunny"],
    "handbags": ["sunny", "rainy", "snowy", "cold", "hot", "all-weather"],
    "harem-pants": ["hot", "warm"],
    "shorts": ["hot"],
    "skirts": ["hot", "warm"],
    "sleepsuit": ["hot"],
    "swimwear": ["hot", "sunny"],
    "tshirts": ["hot", "warm"],
    "tops": ["hot", "warm"],
    "blazers": ["cold", "cool", "windy"],
    "boots": ["cold", "snowy", "freezing"],
    "coats": ["freezing", "snowy", "cold"],
    "jackets": ["cold", "cool", "windy"],
    "nehru-jackets": ["cold", "cool"],
    "shawl": ["cold", "cool"],
    "sweaters": ["cold", "freezing"],
    "sweatshirts": ["cold", "cool"],
    "thermal-bottoms": ["freezing", "cold"],
    "thermal-tops": ["freezing", "cold"],
    "waistcoat": ["cold", "cool"],
    "rain-jacket": ["rainy", "stormy"],
    "bra": ["hot", "warm", "cool", "cold", "all-weather"],
    "bracelet": ["hot", "warm", "cool", "cold", "all-weather"],
    "briefs": ["hot", "warm", "cool", "cold", "all-weather"],
    "burqas": ["hot", "warm", "cool", "cold", "all-weather"],
    "churidar": ["hot", "warm", "cool", "cold", "all-weather"],
    "clothing-set": ["hot", "warm", "cool", "cold", "all-weather"],
    "co-ords": ["hot", "warm", "cool", "cold", "all-weather"],
    "dhotis": ["hot", "warm", "cool", "cold", "all-weather"],
    "dress-material": ["hot", "warm", "cool", "cold", "all-weather"],
    "dungarees": ["cool", "windy"],
    "dupatta": ["hot", "warm", "cool", "cold", "all-weather"],
    "earrings": ["hot", "warm", "cool", "cold", "all-weather"],
    "flats": ["hot", "warm", "cool", "cold", "all-weather"],
    "hair-accessory": ["hot", "warm", "cool", "cold", "all-weather"],
    "heels": ["hot", "warm", "cool", "cold", "all-weather"],
    "jeans": ["hot", "warm", "cool", "cold", "all-weather"],
    "jeggings": ["hot", "warm", "cool", "cold", "all-weather"],
    "jewellery-set": ["hot", "warm", "cool", "cold", "all-weather"],
    "jumpsuit": ["hot", "warm", "cool"],
    "kurta-sets": ["hot", "warm", "cool", "cold", "all-weather"],
    "kurtas": ["hot", "warm", "cool", "cold", "all-weather"],
    "kurtis": ["hot", "warm", "cool", "cold", "all-weather"],
    "leggings": ["hot", "warm", "cool", "cold", "all-weather"],
    "lehenga-choli": ["hot", "warm", "cool", "cold", "all-weather"],
    "lingerie-accessories": ["hot", "warm", "cool", "cold", "all-weather"],
    "lingerie-set": ["hot", "warm", "cool", "cold", "all-weather"],
    "lounge-pants": ["hot", "warm", "cool", "cold", "all-weather"],
    "lounge-shorts": ["hot", "warm"],
    "lounge-tshirts": ["hot", "warm", "cool", "cold", "all-weather"],
    "necklace-and-chains": ["hot", "warm", "cool", "cold", "all-weather"],
    "night-suits": ["hot", "warm", "cool", "cold", "all-weather"],
    "nightdress": ["hot", "warm", "cool", "cold", "all-weather"],
    "outdoor-masks": ["hot", "warm", "cool", "cold", "all-weather"],
    "palazzos": ["hot", "warm", "cool"],
    "patiala": ["hot", "warm", "cool", "cold", "all-weather"],
    "patiala-and-dupatta": ["hot", "warm", "cool", "cold", "all-weather"],
    "pyjamas": ["hot", "warm", "cool", "cold", "all-weather"],
    "robe": ["cold", "freezing"],
    "bath-robe": ["cold", "freezing"],
    "salwar": ["hot", "warm", "cool", "cold", "all-weather"],
    "salwar-and-dupatta": ["hot", "warm", "cool", "cold", "all-weather"],
    "saree-accessories": ["hot", "warm", "cool", "cold", "all-weather"],
    "saree-blouse": ["hot", "warm", "cool", "cold", "all-weather"],
    "sarees": ["hot", "warm", "cool", "cold", "all-weather"],
    "scarves": ["cold", "windy"],
    "shapewear": ["hot", "warm", "cool", "cold", "all-weather"],
    "shirts": ["hot", "warm", "cool", "cold", "all-weather"],
    "shrug": ["cool", "windy"],
    "slips": ["hot", "warm", "cool", "cold", "all-weather"],
    "socks": ["hot", "warm", "cool", "cold", "all-weather"],
    "sports-sandals": ["hot", "warm"],
    "stockings": ["cold", "freezing"],
    "stoles": ["cold", "freezing"],
    "tights": ["cold", "freezing"],
    "track-pants": ["cold", "cool"],
    "tracksuits": ["cold", "cool"],
    "trousers": ["hot", "warm", "cool", "cold", "all-weather"],
    "tunics": ["cool", "windy"]
}
df = pd.read_csv('myntra_dataset.csv')
for index, row in df.iterrows():
    category = row['individual_category']

    seasons = category_seasons.get(category, ["summer", "spring", "fall", "winter", "all-season"])
    weather = category_weather.get(category, ["sunny", "rainy", "snowy", "freezing", "windy", "cool", "cold", "warm", "hot", "stormy", "all-weather"])

    seasons_text = " ".join(seasons)
    weather_text = " ".join(weather)
    df.at[index, 'description'] = f"{row['description']} {seasons_text} {weather_text}"
df.to_csv("myntra_dataset_weather.csv",index=False)


In [ ]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipModel, BlipImageProcessor

# Configurare device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizare device: {device}")

def generate_tags(image_path):
    """Generează descrieri și tag-uri din imagine folosind BLIP"""
    # Încărcare model pentru captioning
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

    # Încărcare imagine
    image = Image.open(image_path).convert('RGB')

    # Procesare imagine și generare descriere
    inputs = processor(image, return_tensors="pt").to(device)
    output = model.generate(**inputs, max_length=30)
    caption = processor.decode(output[0], skip_special_tokens=True)

    # Extragere tag-uri
    words = caption.lower().split()
    stop_words = {'a', 'an', 'the', 'and', 'is', 'are', 'in', 'on', 'at', 'with', 'by', 'of', 'to', 'for'}
    tags = [word for word in words if word not in stop_words and len(word) > 2]

    return {
        'caption': caption,
        'tags': list(set(tags))  # Elimină duplicate
    }

def extract_embeddings(image_path):
    """Extrage embeddings din imagine folosind BLIP"""
    # Încărcare model și procesor pentru embeddings
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

    # Încărcare imagine
    image = Image.open(image_path).convert('RGB')

    # Pentru BlipModel, trebuie să includem și un prompt text gol
    # Acest lucru este diferit de BlipForConditionalGeneration
    text = ""  # Un text gol
    inputs = processor(image, text, return_tensors="pt").to(device)

    # Extrage embeddings
    with torch.no_grad():
        outputs = model(**inputs)

    # Returnare embedding imagine
    image_embeds = outputs.image_embeds
    return image_embeds.cpu().numpy()

# Exemplu de utilizare
if __name__ == "__main__":
    img_path = "kurta.jpeg"

    # Generare tag-uri
    tags_result = generate_tags(img_path)
    print("Descriere:", tags_result['caption'])
    print("Tag-uri:", tags_result['tags'])

    try:
        # Extragere embeddings
        embeddings = extract_embeddings(img_path)
        print("Dimensiune embedding:", embeddings.shape)
    except Exception as e:
        print(f"Eroare la extragerea embeddings: {e}")
        print("Încercăm o abordare alternativă...")

        # Abordare alternativă pentru extragerea embeddings
        processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

        image = Image.open(img_path).convert('RGB')
        inputs = processor(image, return_tensors="pt").to(device)

        # Folosim modelul de captioning pentru a obține embeddings
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        # Obținem ultima stare ascunsă a encoderului de imagine
        last_hidden_state = outputs.encoder_last_hidden_state
        # Calculăm media pe dimensiunea secvenței pentru a obține un singur vector
        image_embedding = last_hidden_state.mean(dim=1)

        print("Dimensiune embedding alternativ:", image_embedding.shape)
        print("Embedding extras cu succes!")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Utilizare device: cpu


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: 'kurta.jpeg'

In [ ]:
!pip install torch torchvision transformers pillow timm fairscale

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fairscale: filename=fairscale-0.4.13-py3-none-any.whl size=332206 sha256=3b0c88cc03dcaef34ed3299346b95b5dbdfcf25f29cc01226061ab5c306cfd67
  Stored in directory: /root/.cache/pip/wheels/95/ef/96/5044bde220b2ea299bdc6ec05051e0ef187fad45b341d1c273
Successfully built fairscale


In [ ]:
from google.colab import files
uploaded = files.upload()